# 🧠 NeuroEvasion — H100 Training Notebook

This notebook walks you through training the co-evolutionary pursuit-evasion DQN agents on a **Google Colab H100** runtime.

**Steps:**
1. GPU sanity check
2. Clone the repo
3. Install dependencies
4. Mount Google Drive (checkpoint persistence)
5. Configure training
6. Run training
7. Monitor with TensorBoard
8. Evaluate trained agents
9. Download final checkpoints

> **Resume support:** If the runtime is preempted, simply re-run from **Cell 1**. Training automatically resumes from the latest Drive-synced checkpoint.

---
## 1 · GPU Sanity Check
Verify that CUDA is available and confirm the GPU model.

In [ ]:
import torch

assert torch.cuda.is_available(), "❌ No GPU detected — change Runtime > Hardware accelerator to GPU (H100)."

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_mem / 1e9

print(f"✅ GPU detected: {gpu_name}")
print(f"   VRAM: {vram_gb:.1f} GB")
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA: {torch.version.cuda}")

---
## 2 · Clone the Repository
Clone **NeuroEvasion** into `/content/NeuroEvasion`.

> If your repo is **private**, replace the URL with an HTTPS token URL or upload the project manually via the Colab file browser.

In [ ]:
import os

REPO_URL = "https://github.com/YOUR_USERNAME/NeuroEvasion.git"  # ← EDIT THIS
PROJECT_DIR = "/content/NeuroEvasion"

if not os.path.isdir(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
    print(f"✅ Cloned into {PROJECT_DIR}")
else:
    # Pull latest changes if already cloned
    !cd {PROJECT_DIR} && git pull
    print(f"✅ Repo already exists — pulled latest changes.")

# Add project root to Python path so all imports resolve
import sys
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

os.chdir(PROJECT_DIR)
print(f"📂 Working directory: {os.getcwd()}")

---
## 3 · Install Dependencies
Colab already ships with PyTorch + CUDA. We only need the remaining packages.

In [ ]:
!pip install -q -r requirements.txt
print("✅ All dependencies installed.")

---
## 4 · Mount Google Drive
Checkpoints are synced here so training survives runtime preemptions.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_CKPT_DIR = "/content/drive/MyDrive/NeuroEvasion/ckpts"
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)

print(f"✅ Google Drive mounted.")
print(f"   Checkpoint sync path: {DRIVE_CKPT_DIR}")

---
## 5 · Configure Training

Tweak any hyperparameters below. The defaults are production-ready for H100.

| Parameter | Default | Notes |
|-----------|---------|-------|
| `num_episodes` | 500,000 | Total training episodes |
| `grid_size` | 20 | N×N game grid |
| `learning_rate` | 1e-4 | Adam LR |
| `batch_size` | 64 | Replay buffer sample size |
| `checkpoint_interval` | 1,000 | Save every N episodes |
| `multi_discrete` | False | Enable dual-head (move + tool) agents |

In [ ]:
from config import Config

config = Config()

# ─── Device ─────────────────────────────────────────────────────────────────
config.training.device = "cuda"

# ─── Checkpoint persistence ─────────────────────────────────────────────────
config.checkpoint.drive_sync_dir = DRIVE_CKPT_DIR
config.checkpoint.interval       = 1_000    # Save every 1 000 episodes
config.checkpoint.keep_last_n    = 5        # Rolling window on local disk

# ─── Training hyperparams (edit as needed) ──────────────────────────────────
config.training.num_episodes  = 500_000
config.game.grid_size         = 20
config.agent.learning_rate    = 1e-4
config.agent.batch_size       = 64
config.seed                   = 42

# ─── Multi-Discrete (optional — set True to enable move + tool actions) ────
# config.multi_discrete.use_multi_discrete = True
# config.multi_discrete.use_dueling        = True

print("✅ Config ready.")
print(f"   Device:     {config.training.device}")
print(f"   Episodes:   {config.training.num_episodes:,}")
print(f"   Grid:       {config.game.grid_size}×{config.game.grid_size}")
print(f"   LR:         {config.agent.learning_rate}")
print(f"   Batch size: {config.agent.batch_size}")
print(f"   Drive sync: {config.checkpoint.drive_sync_dir}")

---
## 6 · Run Training

This starts the co-evolutionary training loop. Progress is printed every 100 episodes.

- **Auto-resume:** If a checkpoint exists (locally or from Drive sync), training resumes automatically.
- **Emergency saves:** `SIGTERM` / `SIGINT` trigger an immediate checkpoint before shutdown.
- **Interrupt safely:** Use `Runtime > Interrupt execution` — an emergency checkpoint is saved first.

In [ ]:
import shutil

# If resuming after a preemption, restore Drive checkpoints to local disk first
local_ckpt = os.path.join(PROJECT_DIR, "checkpoints")
if not os.path.exists(os.path.join(local_ckpt, "manifest.json")) and os.path.exists(
    os.path.join(DRIVE_CKPT_DIR, "manifest.json")
):
    print("📥 Restoring checkpoints from Google Drive …")
    if os.path.isdir(local_ckpt):
        shutil.rmtree(local_ckpt)
    shutil.copytree(DRIVE_CKPT_DIR, local_ckpt)
    print("   ✅ Checkpoints restored.")

# --- Start training ---
from training.trainer import train

train(config)

---
## 7 · Monitor with TensorBoard

Run this cell **while training is running** in another tab, or after training completes, to visualise reward curves, loss, and epsilon decay.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/

---
## 8 · Evaluate Trained Agents

Load the final checkpoints and run 1 000 evaluation games to measure win rates.

In [ ]:
from config import Config as EvalConfig
from evaluation.evaluator import evaluate

eval_cfg = EvalConfig()
eval_cfg.training.device = "cuda"

snake_model = os.path.join(PROJECT_DIR, "checkpoints", "snake_final.pt")
bait_model  = os.path.join(PROJECT_DIR, "checkpoints", "bait_final.pt")

if os.path.exists(snake_model) and os.path.exists(bait_model):
    print("📊 Running evaluation (1 000 games) …")
    evaluate(eval_cfg, snake_model, bait_model, num_games=1000)
else:
    print("⚠️  Final checkpoint files not found yet. Run training first (Cell 6).")
    print(f"   Expected: {snake_model}")
    print(f"   Expected: {bait_model}")

---
## 9 · Download Final Checkpoints

Download the trained model weights to your local machine.

In [ ]:
from google.colab import files

for model_file in ["checkpoints/snake_final.pt", "checkpoints/bait_final.pt"]:
    full_path = os.path.join(PROJECT_DIR, model_file)
    if os.path.exists(full_path):
        files.download(full_path)
        print(f"⬇️  Downloaded: {model_file}")
    else:
        print(f"⚠️  Not found: {model_file} — run training first.")

print("\n✅ Done! Your trained agent weights are also synced to Google Drive at:")
print(f"   {DRIVE_CKPT_DIR}")